# Validar caracteristicas en el diccionario de datos original

Se va a responder:
- ¿La necesitamos?
- ¿Ya encontramos evidencia pública?
- ¿En qué dataset?
- ¿Es directa o derivada?
- ¿La puede capturar la extensión?
- ¿Necesita cálculo?
- ¿La debe evaluar un experto?
- ¿Es X, Y o solamente contexto?

In [35]:
from pathlib import Path

import pandas as pd
import numpy as np


In [36]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

INTERIM_DIR = Path("../data/interim").resolve()
REPORTS_DIR = Path("../reports/tables").resolve()

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(INTERIM_DIR)
print(REPORTS_DIR)

/Users/uriel/Documents/machine_learning/data/interim
/Users/uriel/Documents/machine_learning/reports/tables


In [37]:
a11y = pd.read_csv(
    INTERIM_DIR
    / "a11y_features_enriched.csv"
)

repo01_validation = pd.read_csv(
    INTERIM_DIR
    / "repo01_feature_validation.csv"
)

delfi_validation = pd.read_csv(
    INTERIM_DIR
    / "delfi_feature_validation.csv"
)

irr = pd.read_csv(
    INTERIM_DIR
    / "irr_results.csv"
)

print("A11y:", a11y.shape)
print(
    "Repo01 validation:",
    repo01_validation.shape
)
print(
    "DELFI validation:",
    delfi_validation.shape
)
print(
    "IRR:",
    irr.shape
)

A11y: (1320, 50)
Repo01 validation: (6, 4)
DELFI validation: (12, 8)
IRR: (104, 35)


In [38]:
VARIABLES_TESIS = [

    # -------------------------
    # CONTEXTO
    # -------------------------

    "instance_id",
    "platform",
    "task_type",
    "page_type",

    # -------------------------
    # TIEMPO
    # -------------------------

    "task_time_s",
    "mean_response_time_ms",
    "max_response_time_ms",
    "mean_pre_action_pause_s",

    # -------------------------
    # INTERACCIÓN
    # -------------------------

    "click_count",
    "keypress_count",
    "scroll_count",
    "backtracking_count",
    "repeated_action_count",
    "navigation_steps",

    # -------------------------
    # H1
    # -------------------------

    "feedback_present",
    "feedback_coverage_ratio",
    "feedback_latency_ms",

    "loader_present",
    "loader_duration_ms",

    "progress_indicator_present",
    "status_message_present",
    "completion_feedback_present",

    # -------------------------
    # H4
    # -------------------------

    "component_style_consistency_ratio",
    "css_class_consistency_ratio",
    "navigation_consistency_ratio",
    "label_consistency_ratio",
    "semantic_role_consistency_ratio",
    "dom_structure_similarity",

    # -------------------------
    # H5
    # -------------------------

    "input_validation_present",
    "client_validation_present",
    "validation_error_count",
    "invalid_submission_blocked",
    "confirmation_dialog_present",
    "destructive_action_protected",
    "error_feedback_present",
    "user_error_count",

    # -------------------------
    # H6
    # -------------------------

    "visible_labels_ratio",
    "unlabeled_controls_count",
    "breadcrumbs_present",
    "navigation_persistence_ratio",
    "navigation_depth",
    "visible_options_ratio",

    # -------------------------
    # ISO / RESULTADO
    # -------------------------

    "task_completed",

    # -------------------------
    # TARGETS
    # -------------------------

    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

len(VARIABLES_TESIS)

47

In [39]:
VARIABLES_TESIS.insert(
    VARIABLES_TESIS.index(
        "visible_labels_ratio"
    ),
    "accessible_name_ratio"
)

len(VARIABLES_TESIS)

48

In [40]:
categorias = {

    "instance_id": "CONTEXTO",
    "platform": "CONTEXTO",
    "task_type": "CONTEXTO",
    "page_type": "CONTEXTO",

    "task_time_s": "TEMPORAL",
    "mean_response_time_ms": "TEMPORAL",
    "max_response_time_ms": "TEMPORAL",
    "mean_pre_action_pause_s": "TEMPORAL",

    "click_count": "INTERACCION",
    "keypress_count": "INTERACCION",
    "scroll_count": "INTERACCION",
    "backtracking_count": "INTERACCION",
    "repeated_action_count": "INTERACCION",
    "navigation_steps": "INTERACCION",

    "feedback_present": "H1",
    "feedback_coverage_ratio": "H1",
    "feedback_latency_ms": "H1",
    "loader_present": "H1",
    "loader_duration_ms": "H1",
    "progress_indicator_present": "H1",
    "status_message_present": "H1",
    "completion_feedback_present": "H1",

    "component_style_consistency_ratio": "H4",
    "css_class_consistency_ratio": "H4",
    "navigation_consistency_ratio": "H4",
    "label_consistency_ratio": "H4",
    "semantic_role_consistency_ratio": "H4",
    "dom_structure_similarity": "H4",

    "input_validation_present": "H5",
    "client_validation_present": "H5",
    "validation_error_count": "H5",
    "invalid_submission_blocked": "H5",
    "confirmation_dialog_present": "H5",
    "destructive_action_protected": "H5",
    "error_feedback_present": "H5",
    "user_error_count": "H5",

    "accessible_name_ratio": "H6",
    "visible_labels_ratio": "H6",
    "unlabeled_controls_count": "H6",
    "breadcrumbs_present": "H6",
    "navigation_persistence_ratio": "H6",
    "navigation_depth": "H6",
    "visible_options_ratio": "H6",

    "task_completed": "ISO_CALIDAD_EN_USO",

    "H1_violation": "TARGET",
    "H4_violation": "TARGET",
    "H5_violation": "TARGET",
    "H6_violation": "TARGET"
}

In [41]:
roles = {}

for variable in VARIABLES_TESIS:

    if variable in [
        "H1_violation",
        "H4_violation",
        "H5_violation",
        "H6_violation"
    ]:

        roles[variable] = "Y"

    elif variable in [
        "instance_id"
    ]:

        roles[variable] = "IDENTIFICADOR"

    elif variable in [
        "platform",
        "task_type",
        "page_type"
    ]:

        roles[variable] = "CONTEXTO"

    else:

        roles[variable] = "X"

In [42]:
diccionario = pd.DataFrame({
    "variable": VARIABLES_TESIS
})

diccionario[
    "categoria"
] = diccionario[
    "variable"
].map(categorias)

diccionario[
    "rol_ml"
] = diccionario[
    "variable"
].map(roles)

diccionario.head(20)

,variable,categoria,rol_ml
0,instance_id,CONTEXTO,IDENTIFICADOR
1,platform,CONTEXTO,CONTEXTO
2,task_type,CONTEXTO,CONTEXTO
3,page_type,CONTEXTO,CONTEXTO
4,task_time_s,TEMPORAL,X
5,mean_response_time_ms,TEMPORAL,X
6,max_response_time_ms,TEMPORAL,X
7,mean_pre_action_pause_s,TEMPORAL,X
8,click_count,INTERACCION,X
9,keypress_count,INTERACCION,X


In [43]:
a11y.columns.tolist()

['session_dir',
 'instance_id',
 'task_id',
 'task_title',
 'task_type',
 'task_context',
 'task_completed',
 'started_at',
 'ended_at',
 'task_time_s',
 'source_group',
 'source_type',
 'event_count',
 'click_count',
 'keypress_count',
 'scroll_count',
 'input_count',
 'focus_count',
 'blur_count',
 'page_event_count',
 'unique_pages_visited',
 'navigation_steps',
 'backtracking_count',
 'repeated_action_count',
 'mean_pre_action_pause_s',
 'a11y_snapshot_count',
 'mean_node_count',
 'max_node_count',
 'mean_interactive_controls',
 'total_interactive_controls',
 'total_named_controls',
 'max_unlabeled_controls_count',
 'max_a11y_tree_depth',
 'accessible_name_ratio',
 'html_snapshot_count',
 'mean_html_elements',
 'max_html_elements',
 'mean_interactive_controls_html',
 'total_interactive_controls_html',
 'total_labeled_controls_html',
 'max_unlabeled_controls_html',
 'input_validation_present',
 'progress_indicator_present',
 'status_region_present',
 'breadcrumbs_present',
 'max_dom

In [44]:
a11y_mapping = {

    "instance_id":
        "instance_id",

    "task_type":
        "task_type",

    "task_time_s":
        "task_time_s",

    "click_count":
        "click_count",

    "keypress_count":
        "keypress_count",

    "scroll_count":
        "scroll_count",

    "backtracking_count":
        "backtracking_count",

    "repeated_action_count":
        "repeated_action_count",

    "navigation_steps":
        "navigation_steps",

    "progress_indicator_present":
        "progress_indicator_present",

    "input_validation_present":
        "input_validation_present",

    "breadcrumbs_present":
        "breadcrumbs_present",

    "task_completed":
        "task_completed",

    "accessible_name_ratio":
        "accessible_name_ratio"
}

In [45]:
a11y_proxies = {

    "visible_labels_ratio":
        "control_label_ratio_html",

    "unlabeled_controls_count":
        "max_unlabeled_controls_html",

    "status_message_present":
        "status_region_present"
}

In [46]:
def estado_a11y(variable):

    if variable in a11y_mapping:

        columna = a11y_mapping[
            variable
        ]

        if columna in a11y.columns:
            return "DIRECTA_O_DERIVADA"

    if variable in a11y_proxies:

        columna = a11y_proxies[
            variable
        ]

        if columna in a11y.columns:
            return "PROXY"

    return "NO"

In [47]:
diccionario[
    "a11y_estado"
] = diccionario[
    "variable"
].apply(
    estado_a11y
)

In [48]:
diccionario[
    "a11y_estado"
].value_counts()

a11y_estado
NO                    31
DIRECTA_O_DERIVADA    14
PROXY                  3
Name: count, dtype: int64

In [49]:
repo01_map = (
    repo01_validation
    .set_index(
        "variable_tesis"
    )
    .to_dict(
        orient="index"
    )
)

In [50]:
def repo01_estado(variable):

    if variable in repo01_map:

        return repo01_map[
            variable
        ].get(
            "estado",
            "DISPONIBLE"
        )

    return "NO"

In [51]:
diccionario[
    "repo01_estado"
] = diccionario[
    "variable"
].apply(
    repo01_estado
)

In [52]:
def estado_delfi(variable):

    matches = delfi_validation[
        delfi_validation[
            "variable_tesis_candidata"
        ]
        ==
        variable
    ]

    if len(matches) == 0:
        return "NO"

    tipos = (
        matches[
            "tipo_evidencia"
        ]
        .dropna()
        .unique()
        .tolist()
    )

    return (
        " / ".join(tipos)
    )

In [53]:
diccionario[
    "delfi_estado"
] = diccionario[
    "variable"
].apply(
    estado_delfi
)

In [54]:
diccionario[
    [
        "variable",
        "categoria",
        "a11y_estado",
        "repo01_estado",
        "delfi_estado"
    ]
]

,variable,categoria,a11y_estado,repo01_estado,delfi_estado
0,instance_id,CONTEXTO,DIRECTA_O_DERIVADA,NO,NO
1,platform,CONTEXTO,NO,NO,NO
2,task_type,CONTEXTO,DIRECTA_O_DERIVADA,NO,NO
3,page_type,CONTEXTO,NO,NO,NO
4,task_time_s,TEMPORAL,DIRECTA_O_DERIVADA,NO,NO
5,mean_response_time_ms,TEMPORAL,NO,NO,NO
6,max_response_time_ms,TEMPORAL,NO,NO,NO
7,mean_pre_action_pause_s,TEMPORAL,NO,DERIVABLE_REQUIERE_VALIDACION,NO
8,click_count,INTERACCION,DIRECTA_O_DERIVADA,DERIVABLE_VALIDADA_EN_MUESTRA,NO
9,keypress_count,INTERACCION,DIRECTA_O_DERIVADA,VALIDADA,NO


In [55]:
diccionario[
    "irr_aporte"
] = "NO"

In [56]:
mask_targets = diccionario[
    "rol_ml"
] == "Y"

diccionario.loc[
    mask_targets,
    "irr_aporte"
] = (
    "REFERENCIA_PARA_ESQUEMA_DE_ETIQUETADO"
)

In [57]:
diccionario[
    "requiere_experto"
] = False

diccionario.loc[
    diccionario["rol_ml"] == "Y",
    "requiere_experto"
] = True

In [58]:
extension_directas = {

    "instance_id",
    "platform",
    "task_type",
    "page_type",

    "task_time_s",

    "click_count",
    "keypress_count",
    "scroll_count",

    "feedback_present",
    "feedback_latency_ms",

    "loader_present",
    "loader_duration_ms",

    "progress_indicator_present",
    "status_message_present",
    "completion_feedback_present",

    "input_validation_present",
    "client_validation_present",

    "validation_error_count",

    "confirmation_dialog_present",

    "error_feedback_present",

    "breadcrumbs_present",

    "task_completed"
}

In [59]:
extension_derivadas = {

    "mean_response_time_ms",
    "max_response_time_ms",
    "mean_pre_action_pause_s",

    "backtracking_count",
    "repeated_action_count",
    "navigation_steps",

    "feedback_coverage_ratio",

    "component_style_consistency_ratio",
    "css_class_consistency_ratio",
    "navigation_consistency_ratio",
    "label_consistency_ratio",
    "semantic_role_consistency_ratio",
    "dom_structure_similarity",

    "invalid_submission_blocked",
    "destructive_action_protected",

    "user_error_count",

    "accessible_name_ratio",
    "visible_labels_ratio",
    "unlabeled_controls_count",
    "navigation_persistence_ratio",
    "navigation_depth",
    "visible_options_ratio"
}

In [60]:
def captura_extension(variable):

    if variable in extension_directas:
        return "DIRECTA"

    if variable in extension_derivadas:
        return "DERIVADA"

    if variable.startswith("H"):
        return "NO"

    return "PENDIENTE"

In [61]:
diccionario[
    "captura_extension"
] = diccionario[
    "variable"
].apply(
    captura_extension
)

In [62]:
def estado_final(row):

    if row["rol_ml"] == "Y":

        return "REQUIERE_GROUND_TRUTH_EXPERTO"

    if (
        row["a11y_estado"]
        != "NO"
        or
        row["repo01_estado"]
        != "NO"
        or
        row["delfi_estado"]
        != "NO"
    ):

        return "RESPALDADA_POR_DATOS_PUBLICOS"

    if row[
        "captura_extension"
    ] in [
        "DIRECTA",
        "DERIVADA"
    ]:

        return "RECOLECCION_PROPIA"

    return "REVISAR"

In [63]:
diccionario[
    "estado_final"
] = diccionario.apply(
    estado_final,
    axis=1
)

In [64]:
diccionario[
    "estado_final"
].value_counts()

estado_final
RESPALDADA_POR_DATOS_PUBLICOS    22
RECOLECCION_PROPIA               22
REQUIERE_GROUND_TRUTH_EXPERTO     4
Name: count, dtype: int64

In [65]:
pd.crosstab(
    diccionario[
        "categoria"
    ],
    diccionario[
        "estado_final"
    ]
)

estado_final,RECOLECCION_PROPIA,REQUIERE_GROUND_TRUTH_EXPERTO,RESPALDADA_POR_DATOS_PUBLICOS
categoria,,,
CONTEXTO,2,0,2
H1,5,0,3
H4,6,0,0
H5,5,0,3
H6,2,0,5
INTERACCION,0,0,6
ISO_CALIDAD_EN_USO,0,0,1
TARGET,0,4,0
TEMPORAL,2,0,2


In [66]:
diccionario[
    diccionario[
        "rol_ml"
    ]
    ==
    "Y"
]

,variable,categoria,rol_ml,a11y_estado,repo01_estado,delfi_estado,irr_aporte,requiere_experto,captura_extension,estado_final
44,H1_violation,TARGET,Y,NO,NO,NO,REFERENCIA_PARA_ESQUEMA_DE_ETIQUETADO,True,NO,REQUIERE_GROUND_TRUTH_EXPERTO
45,H4_violation,TARGET,Y,NO,NO,NO,REFERENCIA_PARA_ESQUEMA_DE_ETIQUETADO,True,NO,REQUIERE_GROUND_TRUTH_EXPERTO
46,H5_violation,TARGET,Y,NO,NO,NO,REFERENCIA_PARA_ESQUEMA_DE_ETIQUETADO,True,NO,REQUIERE_GROUND_TRUTH_EXPERTO
47,H6_violation,TARGET,Y,NO,NO,NO,REFERENCIA_PARA_ESQUEMA_DE_ETIQUETADO,True,NO,REQUIERE_GROUND_TRUTH_EXPERTO


In [67]:
diccionario.to_csv(
    REPORTS_DIR
    / "feature_dictionary_validation.csv",
    index=False
)

In [68]:
print(
    (
        REPORTS_DIR
        / "feature_dictionary_validation.csv"
    ).exists()
)

True
